# Семинар 3. Устойчивость линейной модели и Ridge

## Почему немного изменились данные, а коэффициенты стали другими?

Мы уже умеем готовить признаки и обучать линейную регрессию. Теперь посмотрим, что происходит, когда признаки почти повторяют друг друга, и познакомимся с **регуляризацией**.

Сначала — небольшие синтетические примеры: матрицы, графики ошибки и интерактив. Затем вернёмся к домам Ames и заменим последний шаг знакомого `Pipeline` на `Ridge`. В конце самостоятельно подберём его настройку `alpha`.

**План:** мультиколлинеарность → форма ошибки → Ridge → масштаб признаков → выбор `alpha`.

Выполняйте ячейки по порядку. Длинные ячейки построения графиков подготовлены заранее; важные вычисления показаны отдельно. Задания в конце оставлены без решений.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

sns.set_theme(style="darkgrid")

## 1. Два признака — почти одна информация

> **Мультиколлинеарность (multicollinearity)** — точная или приближённая линейная зависимость между признаками модели. Один признак можно точно или приблизительно выразить через другие. При наличии свободного члена учитывается и постоянный столбец единиц.

Например, площадь дома и повторное измерение той же площади. Связь может объединять и несколько признаков, даже без очень высокой корреляции каждой пары. Здесь для наглядности оставим два.

Пусть прогнозы получаются так:

$$\widehat y=w_1x_1+w_2x_2.$$

**Где:** $x_1,x_2$ — столбцы признаков по всем объектам; $w_1,w_2$ — их веса; $\widehat y$ — вектор прогнозов. В синтетическом примере признаки и целевая переменная центрированы: из каждого столбца вычтено его среднее. Поэтому свободный член здесь равен нулю.

Если прибавить к первому весу число $t$, а из второго вычесть его, получим:

$$\Delta\widehat y=t(x_1-x_2).$$

**Где:** $t$ — изменение первого веса и противоположное изменение второго; $\Delta\widehat y$ — изменение прогнозов; $x_1,x_2$ — прежние столбцы признаков.

При одинаковых столбцах прогнозы не изменятся. При почти одинаковых — меняются слабо при небольшом движении в этом направлении. Отдельные веса могут компенсировать друг друга.

**SSE (sum of squared errors)** — сумма квадратов ошибок. Сравним несколько пар весов при одинаковых признаках.

In [ ]:
#@title Заготовка двух признаков с заданной корреляцией { display-mode: "form" }
def make_pair(rho, n=80):
    """Return centered unit-length columns with the requested correlation."""
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    x1 = np.sin(angles)
    x1 = (x1 - x1.mean()) / np.linalg.norm(x1 - x1.mean())
    q = np.cos(angles)
    q = q - q.mean()
    q = q - x1 * (x1 @ q)
    q = q / np.linalg.norm(q)
    x2 = rho * x1 + np.sqrt(max(0.0, 1 - rho**2)) * q
    return np.column_stack([x1, x2]), q

In [ ]:
X_exact, _ = make_pair(rho=1.0)
y_exact = X_exact @ np.array([2.0, 2.0])

weight_pairs = np.array([[2., 2.], [12., -8.], [102., -98.]])
rows = []
for weights in weight_pairs:
    prediction = X_exact @ weights
    rows.append({"w1": weights[0], "w2": weights[1],
                 "SSE": np.sum((y_exact - prediction)**2)})
pd.DataFrame(rows).round(10)

В таблице разные веса дают одинаковые прогнозы и нулевую SSE. Из этих данных нельзя однозначно определить два отдельных веса.

Ниже будем менять степень сходства признаков. В заготовке каждый столбец центрирован и имеет **длину 1**; это делает матрицы проще. Все величины в этой синтетике без физических единиц.

## 2. Что происходит с `X.T @ X`?

В формуле МНК уже встречалась **матрица Грама (Gram matrix)** — матрица скалярных произведений столбцов:

$$G=X^\top X=\begin{pmatrix}x_1^\top x_1&x_1^\top x_2\\x_2^\top x_1&x_2^\top x_2\end{pmatrix}
=\begin{pmatrix}1&\rho\\\rho&1\end{pmatrix}.$$

**Где:** $X$ — матрица из двух столбцов $x_1,x_2$; $\top$ — транспонирование; $G$ — матрица Грама; $\rho$ — корреляция этих столбцов. Последнее равенство верно для центрированных столбцов **единичной длины**.

При $\rho=1$ столбцы совпадают, и определитель $G$ равен нулю. При близкой к 1 корреляции матрица ещё может быть обратимой, но сильно сжимает одно направление:

$$G\begin{pmatrix}1\\-1\end{pmatrix}=(1-\rho)\begin{pmatrix}1\\-1\end{pmatrix},\qquad \det G=1-\rho^2.$$

**Где:** $G$ — матрица выше; $(1,-1)^\top$ — направление изменения двух весов в противоположные стороны; $1-\rho$ — коэффициент сжатия в этом направлении; $\det G$ — определитель матрицы.

При $\rho=0{,}98$ направление сжимается в 50 раз. Чтобы восстановить его, обратное преобразование должно растянуть его в 50 раз. Поэтому небольшое изменение данных способно сильно изменить найденные веса.

> **Неустойчивость (instability):** малое изменение обучающих данных может вызвать большое изменение коэффициентов. Приближённая зависимость не обязательно снижает ранг, а нестабильные веса не обязательно означают столь же нестабильные прогнозы.

Один лишь маленький определитель без учёта масштаба матрицы — ненадёжный способ диагностики. Здесь масштаб столбцов специально зафиксирован.

In [ ]:
X_demo, q_demo = make_pair(rho=0.98)
y_demo = X_demo @ np.array([2.0, 2.0])
G = X_demo.T @ X_demo
direction = np.array([1.0, -1.0])

print("X.T @ X:")
print(G.round(4))
print("Результат для направления [1, -1]:", (G @ direction).round(4))
print("Определитель:", round(np.linalg.det(G), 4))
print("Ранг:", np.linalg.matrix_rank(G))

In [ ]:
angles = np.linspace(0, 2 * np.pi, 400)
circle = np.array([np.cos(angles), np.sin(angles)])
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, rho in zip(axs, [0.0, 0.98, 1.0]):
    X_pair, _ = make_pair(rho)
    transformed = (X_pair.T @ X_pair) @ circle
    ax.plot(*circle, "--", color="gray", label="Единичная окружность")
    ax.plot(*transformed, color="black", label="После умножения на G")
    ax.set(xlabel="Первая координата", ylabel="Вторая координата",
           title=f"Корреляция ρ = {rho:g}", xlim=(-2.1, 2.1), ylim=(-2.1, 2.1))
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Как появляется овраг функции ошибки?

Будем рассматривать ошибку как функцию **весов**, сохраняя данные неизменными:

$$L(w)=\|Xw-y\|_2^2=\sum_{i=1}^{n}(\widehat y_i-y_i)^2.$$

**Где:** $L$ — SSE; $w=(w_1,w_2)^\top$ — веса; $X$ — матрица признаков; $y$ — целевые значения; $\widehat y=Xw$ — прогнозы; $n$ — число объектов; $i$ — номер объекта; $\|\cdot\|_2^2$ — сумма квадратов компонентов вектора.

Каждая точка на плоскости $(w_1,w_2)$ — отдельная модель. Высота над ней — её ошибка. **Линия уровня (level curve)** соединяет наборы весов с одинаковым значением функции.

При близких признаках увеличение одного веса можно компенсировать уменьшением другого. Вдоль этого направления ошибка меняется медленно, а поперёк — быстро. Поверхность вытягивается в **овраг**. При точной зависимости дно становится плоским: минимумов бесконечно много.

На графиках сверху — поверхность, снизу — её линии уровня. Оси показывают веса, а не исходные признаки.

In [ ]:
#@title Функции для построения поверхности ошибки { display-mode: "form" }
def objective_grid(X, y, w1, w2, alpha=0.0):
    """Evaluate SSE plus a squared-weight penalty on a two-weight grid."""
    gram = X.T @ X
    target_products = X.T @ y
    return ((gram[0, 0] + alpha) * w1**2
            + 2 * gram[0, 1] * w1 * w2
            + (gram[1, 1] + alpha) * w2**2
            - 2 * target_products[0] * w1 - 2 * target_products[1] * w2 + y @ y)

In [ ]:
weight_axis = np.linspace(-4, 6, 101)
w1_grid, w2_grid = np.meshgrid(weight_axis, weight_axis)
fig = plt.figure(figsize=(15, 9))
for col, rho in enumerate([0.0, 0.98, 1.0]):
    X_pair, _ = make_pair(rho)
    y_pair = X_pair @ np.array([2.0, 2.0])
    loss = objective_grid(X_pair, y_pair, w1_grid, w2_grid)
    ax = fig.add_subplot(2, 3, col + 1, projection="3d")
    ax.plot_surface(w1_grid, w2_grid, loss, cmap="Blues", alpha=0.9,
                    rstride=4, cstride=4, linewidth=0.2)
    ax.set(xlabel="Вес w1", ylabel="Вес w2", zlabel="",
           title=f"ρ = {rho:g}", zlim=(0, 150))
    ax.view_init(elev=28, azim=-65)
    ax.text2D(0.83, 0.92, "SSE", transform=ax.transAxes, ha="center")
    ax = fig.add_subplot(2, 3, col + 4)
    contours = ax.contour(w1_grid, w2_grid, loss, levels=[0.05, 0.2, 1, 4, 16, 40], colors="black")
    ax.clabel(contours, inline=True, fontsize=8, fmt="%g")
    if rho == 1:
        ax.plot(weight_axis, 4 - weight_axis, "--", color="tab:orange", label="Все минимумы: w1 + w2 = 4")
    else:
        ax.scatter(2, 2, color="tab:orange", label="Минимум")
    ax.set(xlabel="Вес w1", ylabel="Вес w2", xlim=(-4, 6), ylim=(-4, 6), title="Линии одинаковой SSE")
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
fig.subplots_adjust(left=0.05, right=0.90, bottom=0.08, top=0.95, wspace=0.35, hspace=0.20)
plt.show()

Форма ошибки связана с той же матрицей $G$:

$$L(\widehat w+t d)-L(\widehat w)=t^2d^\top Gd=t^2\|Xd\|_2^2.$$

**Где:** $L$ — SSE; $\widehat w$ — любое решение МНК; $d$ — направление изменения весов; $t$ — величина шага; $G=X^\top X$; $X$ — матрица признаков. Равенство выполняется, потому что в решении МНК градиент SSE равен нулю.

Если $Xd$ близок к нулю, движение весов в направлении $d$ слабо меняет прогнозы и ошибку. Поэтому **направлению сжатия матрицы соответствует вытянутое направление линий уровня ошибки**.

## 4. Ridge: добавляем ограничение на величину весов

> **Регуляризация (regularization)** — способ ограничить сложность модели или допустимые решения. В Ridge это делается добавлением штрафа за величину весов к ошибке на обучающих данных.

**Ridge-регрессия (ridge regression)** минимизирует:

$$J(w,b)=\underbrace{\|Xw+b\mathbf{1}-y\|_2^2}_{\text{SSE}}+\underbrace{\alpha\|w\|_2^2}_{\text{штраф}}.$$

**Где:** $J$ — функция, которую минимизируем при обучении; $X$ — матрица признаков; $y$ — целевые значения; $w$ — веса признаков; $b$ — свободный член; $\mathbf{1}$ — вектор единиц; $\alpha\geq0$ — сила регуляризации; $\|w\|_2^2$ — сумма квадратов весов. Свободный член $b$ **не штрафуется**.

Большие компенсирующие друг друга веса теперь увеличивают штраф. Ограничивается общая величина весов: модуль каждого отдельного коэффициента не обязан уменьшаться при каждом увеличении `alpha`.

Мы используем **сумму** квадратов ошибок, как [sklearn.linear_model.Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html). Если записывать ошибку как MSE, коэффициент перед штрафом для той же задачи будет равен $\alpha/n$, где $n$ — число обучающих объектов.

> **Параметры модели (model parameters)** — величины, определяемые по обучающим данным: здесь веса и свободный член. **Гиперпараметр (hyperparameter)** — настройка модели или алгоритма обучения, которую задают до конкретного запуска `fit`. Её можно выбирать вручную или процедурой поиска. В Ridge `alpha` — гиперпараметр: каждый `fit` находит веса при уже заданном `alpha`.

Увеличение `alpha` усиливает штраф. При чрезмерном штрафе модель может **недообучиться (underfitting)** — слишком слабо учитывать зависимость целевой переменной от признаков. Улучшение качества на новых данных нужно проверять отдельно.

### Что изменяется в матричном уравнении?

Для центрированных $X$ и $y$ свободный член равен нулю. Как в первом семинаре, приравняем градиент к нулю:

$$\nabla_wJ=2X^\top(Xw-y)+2\alpha w=0
\quad\Longrightarrow\quad (X^\top X+\alpha I)w=X^\top y.$$

**Где:** $\nabla_wJ$ — вектор производных по весам; $X,y$ — центрированные признаки и целевые значения; $w$ — веса; $\alpha$ — сила штрафа; $I$ — единичная матрица размера «число признаков × число признаков».

К диагонали $X^\top X$ добавили `alpha`. Для нашего примера:

$$G+\alpha I=\begin{pmatrix}1+\alpha&\rho\\\rho&1+\alpha\end{pmatrix}.$$

**Где:** $G=X^\top X$ для двух центрированных столбцов единичной длины; $\rho$ — их корреляция; $\alpha$ — сила регуляризации; $I$ — единичная матрица $2\times2$.

При $\rho=0{,}98$ и $\alpha=0{,}5$ масштаб действия вдоль направления $(1,-1)$ меняется с 0,02 до 0,52, а вдоль $(1,1)$ — с 1,98 до 2,48. Различие между направлениями уменьшается. При `alpha > 0` матрица обратима даже при точной зависимости столбцов.

Для $\alpha>0$ решение имеет вид:

$$\widehat w_{\text{Ridge}}=(X^\top X+\alpha I)^{-1}X^\top y.$$

**Где:** $\widehat w_{\text{Ridge}}$ — найденные веса; остальные обозначения прежние; $^{-1}$ — обратная матрица. На практике решаем уравнение через `np.linalg.solve`, не вычисляя обратную матрицу явно.

In [ ]:
alpha_demo = 0.5
w_manual = np.linalg.solve(G + alpha_demo * np.eye(2), X_demo.T @ y_demo)
model_demo = Ridge(alpha=alpha_demo, fit_intercept=False)
model_demo.fit(X_demo, y_demo)

print(pd.DataFrame({"NumPy": w_manual, "sklearn Ridge": model_demo.coef_}, index=["w1", "w2"]))
print("Решения совпадают:", np.allclose(w_manual, model_demo.coef_))

### Как меняется поверхность?

**Ridge делает более округлыми линии уровня суммарной функции «SSE + штраф».** Исходная SSE при тех же данных остаётся прежней. Минимум суммарной функции обычно смещается.

В интерактиве:

1. При `alpha = 0` меняйте корреляцию от 0 до 1. Сравните форму преобразованной окружности и линии уровня SSE.
2. Поставьте `ρ = 0.999`, `alpha = 0` и нажмите «Изменить y». Изменились целевые значения, признаки остались прежними. Посмотрите на веса.
3. Добавьте штраф и повторите изменение `y`. Сравните движение решений.

Синтетические признаки здесь имеют длину 1. У `StandardScaler` другая нормировка: для непостоянного столбца std = 1 и длина равна $\sqrt n$, где $n$ — число объектов. Поэтому численные значения `alpha` между этой синтетикой и домами напрямую не переносим.

In [ ]:
#@title Геометрия Ridge: признаки, матрица и ошибка { display-mode: "form" }
from IPython.display import HTML, display
from uuid import uuid4

_ridge_geometry_html = r"""<div id="RG_OUTPUT_ROOT" class="rg-widget" data-ridge-widget>
  <h3>Похожие признаки, форма ошибки и Ridge</h3>
  <p class="rg-intro">Синтетические данные: 20 наблюдений. Каждый столбец X центрирован и имеет длину 1. Сначала меняйте ρ, затем немного измените y и добавьте α.</p>
  <div class="rg-controls">
    <label><span>Корреляция ρ <output data-rho-value>0,980</output></span><input data-rho type="range" min="0" max="1" step="0.001" value="0.98" aria-label="Корреляция двух признаков"></label>
    <label><span>Сила штрафа α <output data-alpha-value>0,00</output></span><input data-alpha type="range" min="0" max="1" step="0.01" value="0" aria-label="Сила штрафа Ridge"></label>
    <div class="rg-actions"><button type="button" data-perturb>Изменить y</button><button type="button" data-reset>Сбросить</button><span data-perturb-info>Изменение y: 0</span></div>
  </div>
  <p class="rg-takeaway">Изменение α не меняет X, y и исходную SSE. Оно меняет штраф и минимум функции J.</p>
  <div class="rg-grid">
    <section class="rg-panel">
      <h4><span>1</span> Два признака</h4>
      <p class="rg-caption">Одна точка — одно наблюдение train.</p>
      <svg data-scatter role="img" aria-label="Значения двух центрированных признаков единичной длины"></svg>
      <div class="rg-legend"><span><i class="rg-swatch rg-teal rg-dot"></i> наблюдения</span><span><i class="rg-swatch rg-gray rg-dashed"></i> x₂ = x₁</span></div>
    </section>
    <section class="rg-panel">
      <h4><span>2</span> Действие матриц</h4>
      <p class="rg-caption">Образы точек единичной окружности.</p>
      <svg data-map role="img" aria-label="Единичная окружность и её образы при умножении на G и G плюс альфа I"></svg>
      <div class="rg-legend"><span><i class="rg-swatch rg-gray rg-dashed"></i> окружность</span><span><i class="rg-swatch rg-blue"></i> G</span><span><i class="rg-swatch rg-orange rg-dashed"></i> G + αI</span></div>
      <p class="rg-detail" data-determinants></p>
    </section>
    <section class="rg-panel">
      <h4><span>3</span> Исходная ошибка SSE</h4>
      <p class="rg-caption">Значение SSE в зависимости от весов.</p>
      <svg data-sse role="img" aria-label="Контуры исходной суммы квадратов ошибок и её минимум"></svg>
      <div class="rg-legend"><span><i class="rg-swatch rg-blue"></i> контуры SSE</span><span><i class="rg-swatch rg-blue rg-dot"></i> минимум SSE</span></div>
      <p class="rg-detail" data-sse-detail></p>
    </section>
    <section class="rg-panel">
      <h4><span>4</span> Ошибка со штрафом J</h4>
      <p class="rg-caption">Исходная SSE плюс штраф за веса.</p>
      <svg data-ridge role="img" aria-label="Контуры суммы квадратов ошибок со штрафом Ridge и её минимум"></svg>
      <div class="rg-legend"><span><i class="rg-swatch rg-orange"></i> контуры J</span><span><i class="rg-swatch rg-orange rg-dot"></i> минимум J</span></div>
      <p class="rg-detail" data-ridge-detail></p>
    </section>
  </div>
  <p class="rg-contour-note">На графиках 3–4 одинаковые оси весов. Линии показывают прирост над минимумом своей функции: <b>0,1; 0,4; 1; 2</b>. Светлее — меньший прирост. Часть контуров может продолжаться за рамкой.</p>
  <div class="rg-results" aria-live="polite" aria-atomic="true">
    <div class="rg-result rg-ols-result"><strong>OLS · без штрафа</strong><span class="rg-weights" data-ols-weights></span><span data-ols-sse></span><span data-ols-note class="rg-muted"></span></div>
    <div class="rg-result rg-ridge-result"><strong>Ridge · выбранное α</strong><span class="rg-weights" data-ridge-weights></span><span data-ridge-sse></span><span data-ridge-objective></span></div>
  </div>
  <p class="rg-footnote">Все величины без единиц. Значения α здесь относятся к этому синтетическому примеру с длиной столбцов 1. Кнопка меняет только y: добавляет или вычитает один и тот же небольшой вектор длины 0,05.</p>
  <style>
    .rg-widget { --rg-fg:#243140; --rg-muted:#596779; --rg-bg:#ffffff; --rg-card:#f7f9fc; --rg-border:#d4dde8; --rg-grid:#e5ebf2; --rg-blue:#2364ad; --rg-orange:#b75410; --rg-teal:#147b77; --rg-gray:#788495; width:100%; max-width:1120px; box-sizing:border-box; margin:16px auto; padding:20px; border:1px solid var(--rg-border); border-radius:14px; background:var(--rg-bg); color:var(--rg-fg); font:15px/1.5 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; color-scheme:light dark; }
    .rg-widget * { box-sizing:border-box; }
    .rg-widget h3 { font-size:24px; line-height:1.25; margin:0 0 12px; color:var(--rg-fg); }
    .rg-widget h4 { font-size:17px; line-height:1.35; margin:0; color:var(--rg-fg); display:flex; align-items:center; gap:9px; }
    .rg-widget h4>span { border:1px solid var(--rg-border); border-radius:50%; width:27px; height:27px; flex:none; text-align:center; line-height:25px; font-size:14px; font-weight:600; }
    .rg-widget p { margin:9px 0; }
    .rg-widget .rg-intro { color:var(--rg-muted); max-width:900px; }
    .rg-widget .rg-controls { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:14px 24px; background:var(--rg-card); padding:16px; border:1px solid var(--rg-border); border-radius:10px; margin:18px 0; }
    .rg-widget label { display:block; min-width:0; font-weight:600; }
    .rg-widget label>span { display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; }
    .rg-widget output { font-variant-numeric:tabular-nums; }
    .rg-widget input[type=range] { display:block; width:100%; min-width:0; height:25px; margin:7px 0 0; accent-color:var(--rg-blue); }
    .rg-widget input[data-alpha] { accent-color:var(--rg-orange); }
    .rg-widget button { min-height:40px; border:1px solid var(--rg-border); border-radius:7px; padding:7px 14px; background:var(--rg-bg); color:var(--rg-fg); font:inherit; font-weight:550; cursor:pointer; }
    .rg-widget button:hover { border-color:var(--rg-blue); }
    .rg-widget :is(button,input):focus-visible { outline:3px solid var(--rg-blue); outline-offset:3px; }
    .rg-widget .rg-actions { grid-column:1/-1; display:flex; align-items:center; flex-wrap:wrap; gap:10px; }
    .rg-widget [data-perturb-info] { font-size:14px; color:var(--rg-muted); }
    .rg-widget .rg-takeaway { border-left:3px solid var(--rg-orange); padding:8px 12px; margin-bottom:20px; background:var(--rg-card); }
    .rg-widget .rg-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:16px; }
    .rg-widget .rg-panel { min-width:0; padding:14px; border:1px solid var(--rg-border); border-radius:11px; background:var(--rg-bg); }
    .rg-widget .rg-caption { font-size:14px; color:var(--rg-muted); min-height:21px; margin:8px 0 5px; }
    .rg-widget svg { display:block; width:100%; height:auto; overflow:hidden; }
    .rg-widget svg text { font-family:inherit; fill:var(--rg-muted); font-size:13px; }
    .rg-widget .rg-axis-name { font-size:13px; fill:var(--rg-fg); }
    .rg-widget .rg-axis-line { stroke:var(--rg-gray); stroke-width:1.2; }
    .rg-widget .rg-grid-line { stroke:var(--rg-grid); stroke-width:1; }
    .rg-widget .rg-frame { stroke:var(--rg-border); stroke-width:1; fill:none; }
    .rg-widget .rg-legend { display:flex; flex-wrap:wrap; gap:7px 15px; font-size:13px; margin:2px 0 0; min-height:22px; }
    .rg-widget .rg-legend>span { display:inline-flex; align-items:center; gap:6px; white-space:nowrap; }
    .rg-widget .rg-swatch { display:inline-block; width:20px; height:0; border-top:2px solid currentColor; }
    .rg-widget .rg-blue { color:var(--rg-blue); }
    .rg-widget .rg-orange { color:var(--rg-orange); }
    .rg-widget .rg-teal { color:var(--rg-teal); }
    .rg-widget .rg-gray { color:var(--rg-gray); }
    .rg-widget .rg-dashed { border-top-style:dashed; }
    .rg-widget .rg-dot { width:8px; height:8px; border:0; border-radius:50%; background:currentColor; }
    .rg-widget .rg-detail { min-height:42px; margin-bottom:0; font-size:13px; color:var(--rg-muted); font-variant-numeric:tabular-nums; }
    .rg-widget .rg-contour-note { margin:16px 0; font-size:14px; color:var(--rg-muted); }
    .rg-widget .rg-results { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:14px; }
    .rg-widget .rg-result { display:flex; flex-direction:column; gap:5px; min-width:0; border:1px solid var(--rg-border); border-top:3px solid var(--rg-blue); border-radius:8px; background:var(--rg-card); padding:13px 15px; font-size:14px; font-variant-numeric:tabular-nums; }
    .rg-widget .rg-ridge-result { border-top-color:var(--rg-orange); }
    .rg-widget .rg-weights { font-size:18px; font-weight:650; }
    .rg-widget .rg-muted { color:var(--rg-muted); font-size:13px; }
    .rg-widget .rg-footnote { margin:14px 0 0; font-size:13px; color:var(--rg-muted); }
    @media(prefers-color-scheme:dark) {
      .rg-widget { --rg-fg:#e6edf5; --rg-muted:#acbace; --rg-bg:#141c27; --rg-card:#1b2634; --rg-border:#405169; --rg-grid:#29384c; --rg-blue:#90c3ff; --rg-orange:#ffb379; --rg-teal:#7fdad1; --rg-gray:#91a2b9; }
    }
    @media(max-width:780px) { .rg-widget .rg-grid { grid-template-columns:1fr; } }
    @media(max-width:460px) {
      .rg-widget { padding:12px; font-size:14px; }
      .rg-widget h3 { font-size:21px; }
      .rg-widget .rg-controls { padding:12px; gap:12px; }
      .rg-widget .rg-panel { padding:11px; }
      .rg-widget .rg-results { grid-template-columns:1fr; }
      .rg-widget .rg-actions { gap:8px; }
      .rg-widget button { padding:7px 11px; }
    }
  </style>
  <script>
  (() => {
    'use strict';
    const root = document.currentScript?.closest('[data-ridge-widget]') || document.getElementById('RG_OUTPUT_ROOT');
    const uid = 'rg-' + Math.random().toString(36).slice(2);
    const $ = selector => root.querySelector(selector);
    const rhoInput = $('[data-rho]');
    const alphaInput = $('[data-alpha]');
    const svgs = {scatter:$('[data-scatter]'), map:$('[data-map]'), sse:$('[data-sse]'), ridge:$('[data-ridge]')};
    const ns = 'http://www.w3.org/2000/svg';
    const levels = [0.1, 0.4, 1, 2];
    const sqrt2 = Math.sqrt(2);
    const n = 20;
    const x1 = Array.from({length:n}, (_,i) => Math.sqrt(2/n)*Math.cos(2*Math.PI*i/n));
    const q = Array.from({length:n}, (_,i) => Math.sqrt(2/n)*Math.sin(2*Math.PI*i/n));
    let delta = 0;
    let state;
    let previousWidth = 0;
    const fmt = (v, digits=3) => (Math.abs(v)<1e-12 ? 0 : v).toLocaleString('ru-RU', {maximumFractionDigits:digits});
    const fixed = (v,digits) => v.toLocaleString('ru-RU',{minimumFractionDigits:digits,maximumFractionDigits:digits});

    function element(parent, tag, attributes={}, text) {
      const item = document.createElementNS(ns,tag);
      Object.entries(attributes).forEach(([key,value]) => item.setAttribute(key,String(value)));
      if (text !== undefined) item.textContent = text;
      parent.appendChild(item);
      return item;
    }
    function text(parent,x,y,value,attributes={}) {
      return element(parent,'text',{x,y,'text-anchor':'middle',...attributes},value);
    }
    function path(parent, points, attributes={}) {
      return element(parent,'path',{d:points.map((point,i)=>(i?'L':'M')+point[0].toFixed(4)+' '+point[1].toFixed(4)).join(' '),fill:'none',...attributes});
    }
    function setup(svg, name, low, high, ticks, xLabel, yLabel) {
      svg.replaceChildren();
      const width = Math.max(240,Math.round(svg.getBoundingClientRect().width));
      const left=54, top=14, right=15, bottom=58;
      const side=width-left-right, height=top+side+bottom;
      svg.setAttribute('viewBox','0 0 '+width+' '+height);
      svg.setAttribute('height',height);
      const px=v=>left+(v-low)/(high-low)*side;
      const py=v=>top+side-(v-low)/(high-low)*side;
      const defs=element(svg,'defs');
      const clip=element(defs,'clipPath',{id:uid+'-'+name});
      element(clip,'rect',{x:left,y:top,width:side,height:side});
      ticks.forEach(tick=>{
        element(svg,'line',{x1:px(tick),x2:px(tick),y1:top,y2:top+side,'class':'rg-grid-line'});
        element(svg,'line',{x1:left,x2:left+side,y1:py(tick),y2:py(tick),'class':'rg-grid-line'});
        text(svg,px(tick),top+side+22,fmt(tick,1));
        text(svg,left-9,py(tick)+4,fmt(tick,1),{'text-anchor':'end'});
      });
      if (low<=0 && high>=0) {
        element(svg,'line',{x1:px(0),x2:px(0),y1:top,y2:top+side,'class':'rg-axis-line'});
        element(svg,'line',{x1:left,x2:left+side,y1:py(0),y2:py(0),'class':'rg-axis-line'});
      }
      element(svg,'rect',{x:left,y:top,width:side,height:side,'class':'rg-frame'});
      text(svg,left+side/2,height-8,xLabel,{'class':'rg-axis-name'});
      text(svg,13,top+side/2,yLabel,{'class':'rg-axis-name',transform:'rotate(-90 13 '+(top+side/2)+')'});
      const plot=element(svg,'g',{'clip-path':'url(#'+uid+'-'+name+')'});
      svg.dataset.plot = JSON.stringify({left,top,side,low,high,width,height});
      return {svg,plot,px,py,width,side,low,high};
    }

    function compute(rho,alpha,perturbation) {
      const perpendicular = Math.sqrt(Math.max(0,1-rho*rho));
      const x2=x1.map((value,i)=>rho*value+perpendicular*q[i]);
      const y=x1.map((value,i)=>2*value+2*x2[i]+perturbation*q[i]);
      const g1=2*(1+rho), g2=2*(1+rho)+perturbation*perpendicular;
      const gp=(g1+g2)/sqrt2, gm=(g1-g2)/sqrt2;
      const lp=1+rho, lm=1-rho;
      function solve(penalty) {
        const plus=gp/(lp+penalty);
        const minus=lm+penalty>1e-12 ? gm/(lm+penalty) : 0;
        const weights=[(plus+minus)/sqrt2,(plus-minus)/sqrt2];
        const sse=y.reduce((total,target,i)=>total+(x1[i]*weights[0]+x2[i]*weights[1]-target)**2,0);
        const norm2=weights[0]**2+weights[1]**2;
        return {weights,sse,norm2,objective:sse+penalty*norm2};
      }
      return {rho,alpha,delta:perturbation,x1:[...x1],q:[...q],x2,y,lp,lm,ols:solve(0),ridge:solve(alpha),detG:1-rho*rho,detH:(1+alpha)**2-rho*rho};
    }
    function scatter(data) {
      const frame=setup(svgs.scatter,'scatter',-.4,.4,[-.4,-.2,0,.2,.4],'x₁ (без ед.)','x₂ (без ед.)');
      path(frame.plot,[[frame.px(-.4),frame.py(-.4)],[frame.px(.4),frame.py(.4)]],{stroke:'var(--rg-gray)','stroke-width':1.4,'stroke-dasharray':'5 5'});
      data.x1.forEach((value,i)=>element(frame.plot,'circle',{cx:frame.px(value),cy:frame.py(data.x2[i]),r:4,fill:'var(--rg-teal)',opacity:.85}));
    }
    function matrixImage(data) {
      const frame=setup(svgs.map,'map',-2.5,2.5,[-2,-1,0,1,2],'координата 1 (без ед.)','координата 2 (без ед.)');
      const circle=[], original=[], regularized=[];
      for(let i=0;i<=360;i++) {
        const angle=2*Math.PI*i/360, a=Math.cos(angle), b=Math.sin(angle);
        circle.push([frame.px(a),frame.py(b)]);
        original.push([frame.px(a+data.rho*b),frame.py(data.rho*a+b)]);
        regularized.push([frame.px((1+data.alpha)*a+data.rho*b),frame.py(data.rho*a+(1+data.alpha)*b)]);
      }
      path(frame.plot,circle,{stroke:'var(--rg-gray)','stroke-width':1.5,'stroke-dasharray':'4 4','data-curve':'circle'});
      path(frame.plot,original,{stroke:'var(--rg-blue)','stroke-width':3.6,'data-curve':'G'});
      path(frame.plot,regularized,{stroke:'var(--rg-orange)','stroke-width':2.2,'stroke-dasharray':'7 4','data-curve':'H'});
      $('[data-determinants]').textContent='det G = '+fmt(data.detG,5)+'; det(G + αI) = '+fmt(data.detH,5)+'.'+(data.alpha===0?' При α = 0 образы совпадают.':'');
    }
    function contours(data,kind) {
      const penalty=kind==='ridge' ? data.alpha : 0;
      const result=kind==='ridge' ? data.ridge : data.ols;
      const color=kind==='ridge' ? 'var(--rg-orange)' : 'var(--rg-blue)';
      const narrow=svgs[kind].getBoundingClientRect().width<380;
      const frame=setup(svgs[kind],kind,-4,8,narrow?[-4,0,4,8]:[-4,-2,0,2,4,6,8],'w₁ (без ед.)','w₂ (без ед.)');
      const major=data.lp+penalty, minor=data.lm+penalty;
      const center=result.weights;
      const coordinates=(a,b)=>[frame.px(center[0]+(a+b)/sqrt2),frame.py(center[1]+(a-b)/sqrt2)];
      levels.forEach((level,index)=>{
        const attributes={stroke:color,'stroke-width':1.9,opacity:.36+.19*index,'data-level':level};
        const firstRadius=Math.sqrt(level/major);
        if(minor<=1e-12) {
          [-1,1].forEach(sign=>path(frame.plot,[coordinates(sign*firstRadius,-30),coordinates(sign*firstRadius,30)],attributes));
        } else {
          const secondRadius=Math.sqrt(level/minor), points=[];
          for(let i=0;i<=720;i++) {
            const angle=2*Math.PI*i/720;
            points.push(coordinates(firstRadius*Math.cos(angle),secondRadius*Math.sin(angle)));
          }
          path(frame.plot,points,attributes);
        }
      });
      if(minor<=1e-12) path(frame.plot,[coordinates(0,-30),coordinates(0,30)],{stroke:color,'stroke-width':2.4,'stroke-dasharray':'6 4','data-minimum-line':''});
      element(frame.plot,'circle',{cx:frame.px(center[0]),cy:frame.py(center[1]),r:5.5,fill:color,stroke:'var(--rg-bg)','stroke-width':2,'data-minimum':''});
      const detail=$('[data-'+kind+'-detail]');
      detail.textContent=minor<=1e-12 ? 'Пунктир — прямая минимумов. Отмечен один из них: (2; 2).' : 'Точка отмечает единственный минимум функции.';
    }
    function draw() {
      state=compute(Number(rhoInput.value),Number(alphaInput.value),delta);
      $('[data-rho-value]').textContent=fixed(state.rho,3);
      $('[data-alpha-value]').textContent=fixed(state.alpha,2);
      $('[data-perturb-info]').textContent='Изменение y: '+(delta>0?'+':'')+fmt(delta,3);
      scatter(state); matrixImage(state); contours(state,'sse'); contours(state,'ridge');
      const weights=result=>'w = ('+fmt(result.weights[0])+'; '+fmt(result.weights[1])+')';
      $('[data-ols-weights]').textContent=weights(state.ols);
      $('[data-ridge-weights]').textContent=weights(state.ridge);
      $('[data-ols-sse]').textContent='SSE на train: '+fmt(state.ols.sse,6);
      $('[data-ridge-sse]').textContent='SSE на train: '+fmt(state.ridge.sse,6);
      $('[data-ridge-objective]').textContent='J на train: '+fmt(state.ridge.objective,6);
      $('[data-ols-note]').textContent=state.rho===1 ? 'Из нескольких решений выбрано решение с наименьшей длиной w.' : 'Коэффициенты минимизируют исходную SSE.';
      root.dataset.state=JSON.stringify(state);
    }
    rhoInput.addEventListener('input',draw);
    alphaInput.addEventListener('input',draw);
    $('[data-perturb]').addEventListener('click',()=>{delta=delta>0?-.05:.05;draw();});
    $('[data-reset]').addEventListener('click',()=>{rhoInput.value='.98';alphaInput.value='0';delta=0;draw();});
    const observer=new ResizeObserver(()=>{
      const width=root.getBoundingClientRect().width;
      if(Math.abs(width-previousWidth)>.5) {previousWidth=width;draw();}
    });
    observer.observe(root);
    root.ridgeGeometry={getState:()=>JSON.parse(JSON.stringify(state)),compute};
    draw();
  })();
  </script>
</div>
"""
display(HTML(_ridge_geometry_html.replace("RG_OUTPUT_ROOT", "ridge-" + uuid4().hex)))

## 5. Проверяем устойчивость в Python

Зафиксируем признаки с корреляцией 0,999 и несколько раз немного изменим только `y`. Сравним веса и прогноз для **одного и того же нового объекта**. Так мы отделим нестабильность коэффициентов от нестабильности прогнозов.

In [ ]:
X_stability, q_stability = make_pair(rho=0.999)
y_clean = X_stability @ np.array([2.0, 2.0])
new_object = np.array([[0.1, 0.1]])
records = []
for delta in np.linspace(-0.05, 0.05, 21):
    y_changed = y_clean + delta * q_stability
    for name, estimator in [
        ("МНК", LinearRegression(fit_intercept=False)),
        ("Ridge, alpha=0.2", Ridge(alpha=0.2, fit_intercept=False)),
    ]:
        estimator.fit(X_stability, y_changed)
        records.append({"Модель": name, "Изменение y: delta": delta,
                        "w1": estimator.coef_[0], "w2": estimator.coef_[1],
                        "Прогноз": estimator.predict(new_object)[0]})
stability = pd.DataFrame(records)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=stability, x="w1", y="w2", hue="Модель", style="Модель", ax=axs[0])
axs[0].set(xlabel="Вес w1", ylabel="Вес w2", title="Веса при небольших изменениях y")
axs[0].set_aspect("equal", adjustable="datalim")
sns.lineplot(data=stability, x="Изменение y: delta", y="Прогноз", hue="Модель", ax=axs[1])
axs[1].axhline(0.4, color="black", linestyle="--", label="Истинное значение для этого объекта")
axs[1].set(xlabel="delta — величина добавки к y", ylabel="Прогноз для x1 = x2 = 0.1",
           title="Тот же объект при каждом обучении")
axs[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Слева веса МНК заметно меняются, компенсируя друг друга. Справа его прогноз для этого объекта остаётся близким к прежнему. Ridge делает веса устойчивее, но сам прогноз может сместиться относительно истинного значения.

> **Устойчивость коэффициентов и точность прогноза — разные свойства.** Регуляризация может помочь на новых данных; это проверяют по отложенной выборке, а не только по уменьшению весов.

## 6. Что означает картинка «эллипсы и круг»?

Ту же идею Ridge можно представить через ограничение: ищем наименьшую **исходную SSE**, разрешая только веса внутри круга:

$$w_1^2+w_2^2\le R^2.$$

**Где:** $w_1,w_2$ — два веса; $R\geq0$ — допустимая длина вектора весов. Это другой способ задать ограничение; $R$ не равен `alpha`. Для заданного Ridge-решения можно подобрать соответствующий радиус.

Если обычный минимум снаружи, решение с ограничением находится на границе: там круг касается линии уровня ошибки. Если обычный минимум уже внутри, ограничение не мешает его выбрать.

В **Lasso-регрессии (lasso regression)** штраф использует сумму модулей весов. Соответствующая область для двух весов — ромб:

$$|w_1|+|w_2|\le R.$$

**Где:** $|w_1|,|w_2|$ — модули весов; $R$ — допустимая сумма модулей. В вершинах ромба один вес равен нулю. Поэтому Lasso может получать точно нулевые коэффициенты. Здесь это только геометрическое сравнение; практическая модель занятия — Ridge.

**На этой картинке эллипсы исходной SSE не меняются.** Добавлена область допустимых весов. Это отличается от картинки линий уровня суммы «SSE + штраф» в интерактиве.

In [ ]:
#@title Эллипсы исходной ошибки, круг Ridge и ромб Lasso { display-mode: "form" }
example_gram = np.array([[1.0, 0.6], [0.6, 1.0]])
ols_point = np.array([2.0, 0.5])
radius = 1.4
left, right = 0.0, 100.0
for _ in range(70):
    penalty = (left + right) / 2
    candidate = np.linalg.solve(example_gram + penalty * np.eye(2), example_gram @ ols_point)
    if np.linalg.norm(candidate) > radius:
        left = penalty
    else:
        right = penalty
ridge_point = candidate
lasso_point = np.array([radius, 0.0])

axis = np.linspace(-2.5, 3.5, 250)
u, v = np.meshgrid(axis, axis)
d1, d2 = u - ols_point[0], v - ols_point[1]
original_error = d1**2 + 1.2 * d1 * d2 + d2**2
levels = sorted(set([0.1, 0.4, 1, 2, 4, 8] + [
    float((point - ols_point) @ example_gram @ (point - ols_point))
    for point in [ridge_point, lasso_point]]))
t = np.linspace(0, 2 * np.pi, 400)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, point, boundary in [
    (axs[0], "Ridge: ограничена длина весов", ridge_point,
     radius * np.array([np.cos(t), np.sin(t)])),
    (axs[1], "Lasso: ограничена сумма модулей", lasso_point,
     radius * np.array([[1, 0, -1, 0, 1], [0, 1, 0, -1, 0]])),
]:
    ax.contour(u, v, original_error, levels=levels, colors="gray", linewidths=0.8)
    ax.fill(*boundary, color="tab:blue", alpha=0.12, label="Допустимые веса")
    ax.plot(*boundary, color="tab:blue")
    ax.scatter(*ols_point, marker="x", color="black", label="Минимум без ограничения")
    ax.scatter(*point, color="tab:orange", label="Минимум с ограничением")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.set(xlabel="Вес w1", ylabel="Вес w2", xlim=(-2.5, 3.5), ylim=(-2.5, 3.5), title=title)
    ax.set_aspect("equal")
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 7. Почему теперь важен StandardScaler?

В прошлом семинаре стандартизация почти не изменила прогнозы обычного МНК. Но штраф Ridge зависит от величины коэффициентов.

Если умножить признак на 1000, для прежнего вклада в прогноз его вес нужно разделить на 1000. Квадрат этого веса уменьшится в миллион раз. Поэтому при том же `alpha` модель в других единицах решает уже иначе взвешенную задачу.

Сравним прогнозы для одних и тех же объектов: в исходных единицах и после умножения второго признака на 1000. Затем повторим с `StandardScaler` перед Ridge. Скейлер обучается только на обучающих объектах.

`clone(estimator)` создаёт новый необученный объект с теми же настройками. Поэтому в каждом опыте обучается собственная подготовка.

In [ ]:
rng_scale = np.random.default_rng(7)
X_scale_train = rng_scale.normal(size=(80, 2))
X_scale_new = rng_scale.normal(size=(40, 2))
y_scale_train = X_scale_train @ np.array([3.0, 5.0]) + rng_scale.normal(0, 0.3, 80)
unit_change = np.array([1.0, 1000.0])
scale_rows = []
scale_predictions = {}

for name, estimator in [
    ("Без скейлера", Ridge(alpha=80)),
    ("StandardScaler → Ridge", Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=80))])),
]:
    original = clone(estimator).fit(X_scale_train, y_scale_train)
    converted = clone(estimator).fit(X_scale_train * unit_change, y_scale_train)
    pred_original = original.predict(X_scale_new)
    pred_converted = converted.predict(X_scale_new * unit_change)
    scale_predictions[name] = (pred_original, pred_converted)
    scale_rows.append({"Подготовка": name,
                       "Максимальное изменение прогноза": np.max(np.abs(pred_original - pred_converted))})

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (pred_original, pred_converted)) in zip(axs, scale_predictions.items()):
    ax.scatter(pred_original, pred_converted, color="black", alpha=0.4)
    limits = [min(pred_original.min(), pred_converted.min()), max(pred_original.max(), pred_converted.max())]
    ax.plot(limits, limits, "--", color="gray", label="Прогнозы совпадают")
    ax.set(xlabel="Прогноз в исходных единицах признаков",
           ylabel="Прогноз после смены единиц признака", title=name)
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
pd.DataFrame(scale_rows)

После стандартизации смена единиц почти не меняет прогнозы: различие остаётся на уровне погрешности вычислений. Это не обещание, что любой скейлер даст одинаковое качество. Способ масштабирования определяет, как штраф действует на веса.

## 8. Возвращаемся к домам: заменяем один шаг

Используем тот же [датасет Ames / House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data), подготовку и разбиение, что в предыдущем семинаре.

| Признак | Смысл | Единицы после подготовки |
|---|---|---|
| `GrLivArea` | Надземная жилая площадь | м² |
| `GarageArea` | Площадь гаража | м² |
| `TotalBsmtSF` | Площадь подвала | м² |
| `YearBuilt` | Год постройки | календарный год |
| `LotFrontage` | Длина границы участка вдоль улицы | м |
| `Neighborhood` | Район | категория |

Целевая переменная `SalePrice` — цена продажи в долларах США. Код загрузки и подготовки уже дан. Из копии исходных данных переводим площади из квадратных футов в м², а длину — из футов в метры.

In [ ]:
DATA_URL = (
    "https://www.dropbox.com/scl/fi/65u0zpc8dweb9rx1tlzol/train_sem2.csv"
    "?rlkey=5vi4mt47fs6magpldgj5wp2u9&dl=1"
)
raw_data = pd.read_csv(DATA_URL)
print(f"Загружено {len(raw_data)} домов.")

In [ ]:
BASE_FEATURES = ["GrLivArea", "GarageArea", "TotalBsmtSF"]
numeric_features = BASE_FEATURES + ["YearBuilt", "LotFrontage"]
categorical_features = ["Neighborhood"]
features = numeric_features + categorical_features

data = raw_data.copy()
data[BASE_FEATURES + ["LotArea"]] *= 0.09290304
data["LotFrontage"] *= 0.3048
assert (data["SalePrice"] > 0).all()

Сохраняем **те же дома в train, validation и test**, что и в первом семинаре: 60%, 20% и 20%. `random_state` и порядок строк не меняем.

* На **train** обучаем и подготовку признаков, и саму модель.
* На **validation** сравниваем варианты по метрикам.
* **Test** остаётся отложенным до окончательной проверки в конце модуля.

В этом занятии основная метрика выбора — MAE. RMSE и MAPE помогают рассмотреть ошибки с других сторон.

In [ ]:
train_val, test = train_test_split(data, test_size=0.2, random_state=42)
train, validation = train_test_split(train_val, test_size=0.25, random_state=42)

train = train.copy()
validation = validation.copy()
test = test.copy()
y_train = train["SalePrice"]
y_val = validation["SalePrice"]

print(f"Train: {len(train)}; validation: {len(validation)}; test: {len(test)}.")
print("Test отложен: сегодня его не исследуем и не оцениваем.")

**Основная метрика выбора — MAE на validation.** RMSE сильнее реагирует на крупные ошибки, MAPE показывает среднюю абсолютную относительную ошибку.

MAPE выводим в процентах: результат `mean_absolute_percentage_error` умножаем на 100. При нулевом целевом значении обычная формула не определена; при близких к нулю значениях нестабильна. Здесь цены положительны. Универсального порога «хорошей MAPE» нет.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE, $": mean_absolute_error(y_true, y_pred),
        "RMSE, $": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE, %": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
     categorical_features),
])

Для каждого эксперимента создадим собственную цепочку. `clone(preprocessor)` копирует настройки подготовки без обученных статистик.

Числа заполняем и стандартизируем; индикаторы районов оставляем в масштабе 0/1. Сохраняем `drop="first"` из прошлого семинара, чтобы сейчас менять именно модель. Ridge способен работать и с полным набором индикаторов, но изменение кодирования меняет и смысл штрафа.

Как и раньше, неизвестный в train район получает нулевые индикаторы при `handle_unknown="ignore"`; при прогнозе возможно предупреждение. Это не означает, что модель знает цены в таком районе.

In [ ]:
def make_model(estimator):
    return Pipeline([
        ("preprocess", clone(preprocessor)),
        ("regression", estimator),
    ])

Сначала повторим обычную регрессию. Затем попробуем `Ridge(alpha=10)`. Число 10 здесь выбрано **для демонстрации запуска**, а не по результатам поиска лучшей настройки. Подбором займётесь в самостоятельной работе.

Оба `fit` видят только train. При `predict` вся подготовка внутри Pipeline использует сохранённые статистики и категории.

Здесь `Ridge` обучает свободный член автоматически (`fit_intercept=True`); вручную центрировать целевую переменную не нужно.

In [ ]:
ols_model = make_model(LinearRegression())
ols_model.fit(train[features], y_train)
ols_val_pred = ols_model.predict(validation[features])
ols_metrics = regression_metrics(y_val, ols_val_pred)

ridge_model = make_model(Ridge(alpha=10.0))
ridge_model.fit(train[features], y_train)
ridge_val_pred = ridge_model.predict(validation[features])
ridge_metrics = regression_metrics(y_val, ridge_val_pred)

comparison = pd.DataFrame([
    {"Модель": "МНК", **ols_metrics},
    {"Модель": "Ridge, alpha=10", **ridge_metrics},
])
comparison.round(2)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, prediction in zip(axs, ["МНК", "Ridge, alpha=10"], [ols_val_pred, ridge_val_pred]):
    ax.scatter(y_val, prediction, color="black", alpha=0.2)
    limits = [min(y_val.min(), prediction.min()), max(y_val.max(), prediction.max())]
    ax.plot(limits, limits, "--", color="gray", label="Точный прогноз")
    ax.set(xlabel="Фактическая цена, доллары США", ylabel="Прогноз цены, доллары США", title=name)
    ax.legend()
plt.tight_layout()
plt.show()

**Обсудим:** что изменилось по MAE, RMSE и MAPE? Можно ли назвать выбранное `alpha` удачным? Достаточно ли самого факта регуляризации, чтобы ожидать лучшего прогноза?

Настройку выбираем по **предсказательной ошибке на validation**. Штраф входит в функцию обучения, но мы не добавляем его к MAE, RMSE или MAPE при оценке качества.

## 9. Как выбирают гиперпараметр?

Каждому значению `alpha` соответствует отдельное обучение:

```text
alpha → новый Pipeline → fit на train → predict на validation → MAE
```

Сравниваем заранее выбранный набор значений и оставляем вариант с наименьшей MAE на validation. **Выбор по ошибке train не показывает, какой вариант лучше работает на новых объектах.**

В имени параметра `regression__alpha` часть до двойного подчёркивания — имя шага Pipeline, часть после — его настройка. Например:

```python
candidate = clone(ridge_model)
candidate.set_params(regression__alpha=100.0)
```

Это создаёт необученного кандидата и задаёт гиперпараметр. Затем нужны `fit`, `predict` и оценка. Двойное подчёркивание — способ обращаться к настройкам вложенных шагов sklearn.

После выбора `alpha` validation уже участвовала в подборе. Для окончательной проверки у нас остаётся отдельный test; в этом семинаре его продолжаем хранить закрытым.

## 10. Самостоятельная работа

### Задача 1. Подберите alpha

Сравните Ridge для значений `alpha` из подготовленного массива. Признаки, разбиение и подготовку данных оставьте прежними.

1. Для каждого `alpha` создайте нового кандидата, задайте настройку и обучите его на train.
2. Получите прогнозы для train и validation. Сохраните MAE на обеих выборках, а также RMSE и MAPE на validation.
3. Запишите результаты в таблицу `search_results` с указанными ниже названиями столбцов.
4. Найдите `best_alpha` по **наименьшей MAE на validation**. При точном равенстве выберите меньшее значение.
5. Сравните выбранный вариант с МНК. Совпадает ли выбор по MAE с выбором по RMSE?

`np.logspace(-2, 4, 13)` создаёт 13 положительных значений от 0,01 до 10 000 с одинаковым шагом по логарифму.

In [ ]:
alphas = np.logspace(-2, 4, 13)
rows = []
search_results = None
best_alpha = None

# TODO: create and fit a fresh candidate for every alpha.
# TODO: collect train MAE and validation MAE, RMSE, MAPE (percent).
# Use columns: "alpha", "MAE train, $", "MAE validation, $",
#              "RMSE validation, $", "MAPE validation, %".
# TODO: build search_results and choose best_alpha using validation MAE.

### Задача 2. Прочитайте график подбора

После заполнения таблицы выполните готовую ячейку ниже. По оси `alpha` используется логарифмическая шкала: одинаковое расстояние означает одинаковый множитель.

Объясните, что происходит с ошибками train и validation при большом штрафе. Есть ли диапазон, в котором модель теряет качество на обеих выборках? Почему минимум ошибки train не заменяет выбор по validation?

In [ ]:
if search_results is None:
    print("Сначала заполните search_results в задаче 1.")
else:
    ordered_results = search_results.sort_values("alpha")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(ordered_results["alpha"], ordered_results["MAE train, $"], marker="o", label="Train")
    ax.plot(ordered_results["alpha"], ordered_results["MAE validation, $"], marker="o", label="Validation")
    ax.axhline(ols_metrics["MAE, $"], color="black", linestyle="--", label="МНК: validation")
    if best_alpha is not None:
        ax.axvline(best_alpha, color="gray", linestyle=":", label="Выбранное alpha")
    ax.set_xscale("log")
    ax.set(xlabel="alpha — сила регуляризации (логарифмическая шкала)", ylabel="MAE, доллары США")
    ax.legend()
    plt.tight_layout()
    plt.show()

**Ваш вывод:** …

### Задача 3. Зафиксируйте выбор

Создайте `selected_model` с выбранным `best_alpha`, обучите на прежнем train и сохраните три метрики validation в `selected_metrics`. Test не используйте.

В выводе укажите выбранный гиперпараметр, три метрики, сравнение с МНК и причину выбора. Если улучшение небольшое или Ridge проиграл, так и напишите. Один результат на фиксированном разбиении не доказывает, что преимущество повторится на любых новых данных.

Объясните одним предложением: что в вашей модели является **параметрами**, а что — **гиперпараметром**?

In [ ]:
selected_model = None
selected_metrics = None

# TODO: create and fit a pipeline with best_alpha, then evaluate on validation.

**Ваш вывод:** …

## Что мы теперь умеем

Мультиколлинеарность объясняет, почему веса могут компенсировать друг друга и быть нестабильными. Геометрически это связано со сжатием направления матрицей $X^\top X$ и вытянутыми линиями уровня SSE.

Ridge добавляет штраф за сумму квадратов весов. При положительном `alpha` к диагонали матрицы нормального уравнения добавляется положительное число, и решение становится устойчивее к почти зависимым признакам. Качество прогнозов проверяем отдельно.

**Подготовка на train → несколько значений гиперпараметра → сравнение по validation → зафиксированный выбор.**

На следующем шаге можно обсудить, насколько результат выбора зависит от разбиения данных, — это подводка к кросс-валидации.

## Источники

* [sklearn.linear_model.Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) — функция обучения, `alpha`, свободный член и решатели.
* [Руководство sklearn: МНК и Ridge](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification) — коллинеарность и регуляризация.
* [График коэффициентов Ridge при изменении alpha](https://scikit-learn.org/stable/auto_examples/linear_model/plot_ridge_path.html).
* [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) и [clone](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html) — цепочки и новые необученные кандидаты.
* [Разделение данных и выбор модели](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-evaluating-estimator-performance).
* [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) — штраф по сумме модулей весов.
* [House Prices: данные и описание признаков](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).